# 06 Batch Normalization

Part 3: BatchNorm katmanı implementasyonu, eğitimde batch istatistiği, çıkarımda running mean/std, BatchNorm'lu vs BatchNorm'suz karşılaştırma.


In [ ]:
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

def build_dataset(words, block_size=3):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])

n_emb = 10
n_hidden = 200

# 1. BatchNorm'lu Model
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, n_emb), generator=g, requires_grad=True)
W1 = torch.randn((n_emb * 3, n_hidden), generator=g) * (5/3) / ((n_emb * 3)**0.5)
W1.requires_grad = True
# b1 BatchNorm ortalamayi cikardigi icin gereksizdir
W2 = torch.randn((n_hidden, 27), generator=g) * 0.01
W2.requires_grad = True
b2 = torch.randn(27, generator=g) * 0
b2.requires_grad = True

bngain = torch.ones((1, n_hidden), requires_grad=True)
bnbias = torch.zeros((1, n_hidden), requires_grad=True)
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters_bn = [C, W1, W2, b2, bngain, bnbias]

# 2. BatchNorm'suz Model (Baseline)
g_nb = torch.Generator().manual_seed(2147483647)
C_nb = torch.randn((27, n_emb), generator=g_nb, requires_grad=True)
W1_nb = torch.randn((n_emb * 3, n_hidden), generator=g_nb) * (5/3) / ((n_emb * 3)**0.5)
W1_nb.requires_grad = True
b1_nb = torch.randn(n_hidden, generator=g_nb) * 0.01
b1_nb.requires_grad = True
W2_nb = torch.randn((n_hidden, 27), generator=g_nb) * 0.01
W2_nb.requires_grad = True
b2_nb = torch.randn(27, generator=g_nb) * 0
b2_nb.requires_grad = True

parameters_nb = [C_nb, W1_nb, b1_nb, W2_nb, b2_nb]

for i in range(20000):
    ix = torch.randint(0, Xtr.shape[0], (64,))
    
    # BatchNorm ileri gecis
    emb = C[Xtr[ix]].view(-1, n_emb * 3)
    hpreact = emb @ W1
    bnmean = hpreact.mean(0, keepdim=True)
    bnstd = hpreact.std(0, keepdim=True)
    hpreact_norm = (hpreact - bnmean) / (bnstd + 1e-5) * bngain + bnbias
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmean
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstd
    h = torch.tanh(hpreact_norm)
    logits = h @ W2 + b2
    loss_bn = F.cross_entropy(logits, Ytr[ix])
    
    for p in parameters_bn:
        p.grad = None
    loss_bn.backward()
    lr = 0.1 if i < 15000 else 0.01
    for p in parameters_bn:
        p.data += -lr * p.grad
        
    # BatchNorm'suz ileri gecis
    emb_nb = C_nb[Xtr[ix]].view(-1, n_emb * 3)
    h_nb = torch.tanh(emb_nb @ W1_nb + b1_nb)
    logits_nb = h_nb @ W2_nb + b2_nb
    loss_nb = F.cross_entropy(logits_nb, Ytr[ix])
    
    for p in parameters_nb:
        p.grad = None
    loss_nb.backward()
    for p in parameters_nb:
        p.data += -lr * p.grad

# Dev seti degerlendirmesi (cikarimda running istatistikler kullanilir)
emb_dev = C[Xdev].view(-1, n_emb * 3)
hpreact_dev = emb_dev @ W1
hpreact_dev_norm = (hpreact_dev - bnmean_running) / (bnstd_running + 1e-5) * bngain + bnbias
h_dev = torch.tanh(hpreact_dev_norm)
logits_dev = h_dev @ W2 + b2
dev_loss_bn = F.cross_entropy(logits_dev, Ydev)

emb_dev_nb = C_nb[Xdev].view(-1, n_emb * 3)
h_dev_nb = torch.tanh(emb_dev_nb @ W1_nb + b1_nb)
logits_dev_nb = h_dev_nb @ W2_nb + b2_nb
dev_loss_nb = F.cross_entropy(logits_dev_nb, Ydev)

print("BatchNorm ile Dev Loss:", dev_loss_bn.item())
print("BatchNorm olmadan Dev Loss:", dev_loss_nb.item())
